# Random Forests: Ensemble Learning

## Introducción

Los Random Forests son un **método de ensemble learning** que combina múltiples decision trees para crear un modelo más robusto y preciso.

Este notebook explora:
- Cómo los Random Forests superan el problema de overfitting de los decision trees individuales
- Las fuentes clave de aleatoriedad que hacen que el método funcione
- Random Forests para clasificación y regresión
- Análisis de importancia de features
- Hiperparámetros y buenas prácticas

**Prerrequisitos:** Comprensión de los decision trees (ver [decision_tree.ipynb](decision_tree.ipynb))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

## Random Forests: Ensemble Learning


### Introducción a los Random Forests

Como vimos anteriormente, los decision trees tienen una limitación importante: tienden al **overfitting** de los datos de training, especialmente cuando se les permite crecer en profundidad. Un único decision tree tiene **alta varianza** — pequeños cambios en los datos de training pueden resultar en estructuras de árbol muy diferentes.

Los **Random Forests** resuelven este problema usando una técnica de **ensemble learning**. En lugar de depender de un único decision tree, los Random Forests construyen muchos árboles y combinan sus predicciones. Esto se basa en el principio de que "la sabiduría de la multitud" suele ser más precisa que cualquier individuo.

**Idea clave:** Entrenar múltiples decision trees en diferentes subconjuntos aleatorios de datos y features, luego agregar sus predicciones:
- **Clasificación:** Usar votación mayoritaria (la clase que recibe más votos)
- **Regresión:** Promediar las predicciones de todos los árboles

Este enfoque reduce drásticamente el overfitting mientras mantiene los beneficios de interpretabilidad de los decision trees.


### Cómo funcionan los Random Forests

Los Random Forests usan dos fuentes clave de aleatoriedad para crear árboles diversos:

1. **Bootstrap Aggregating (Bagging):**
   - Cada árbol se entrena con una muestra aleatoria de los datos de training (con reemplazo)
   - Esto se denomina muestreo bootstrap
   - Típicamente, cada muestra contiene aproximadamente el 63% de los datos originales

2. **Selección aleatoria de features:**
   - En cada división de un árbol, solo se considera un subconjunto aleatorio de features
   - Esto asegura que los árboles sean diferentes entre sí
   - Valores típicos: $\sqrt{n_{features}}$ para clasificación, $n_{features}/3$ para regresión

**Pasos del algoritmo:**
1. Crear N muestras bootstrap a partir de los datos de training
2. Para cada muestra, entrenar un decision tree:
   - En cada nodo, seleccionar un subconjunto aleatorio de features
   - Elegir la mejor división de este subconjunto
3. Para hacer una predicción:
   - Clasificación: Cada árbol vota, gana la mayoría
   - Regresión: Promediar todas las predicciones de los árboles

![Ilustración de Random Forest](../img/random_forest.png)


### Random Forest para clasificación


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Usando el dataset iris
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data
y = iris.target

# Dividir los datos en training y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar un único decision tree
single_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

# Entrenar un Random Forest con 100 árboles
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

# Comparar predicciones
tree_pred = single_tree.predict(X_test)
rf_pred = rf_classifier.predict(X_test)

print("Accuracy del Decision Tree individual:", accuracy_score(y_test, tree_pred))
print("Accuracy del Random Forest:", accuracy_score(y_test, rf_pred))
print("\nInforme de clasificación del Random Forest:")
print(classification_report(y_test, rf_pred, target_names=iris.target_names))


### Random Forest para regresión

Comparemos un único árbol de regresión con un Random Forest en el dataset de salarios:


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Cargar datos de salarios
df = pd.read_csv("data/salaries.csv")

# Entrenar modelos
tree_reg_full = DecisionTreeRegressor(random_state=42).fit(df[["YearsExperience"]], df["Salary"])
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42).fit(df[["YearsExperience"]], df["Salary"])

# Crear puntos de predicción
x_plot = np.linspace(0, 12, 1000)
tree_predictions = tree_reg_full.predict(x_plot.reshape(-1, 1))
rf_predictions = rf_regressor.predict(x_plot.reshape(-1, 1))

# Visualizar
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(df['YearsExperience'], df['Salary'], label='Datos reales', alpha=0.7)
plt.plot(x_plot, tree_predictions, color='red', label='Decision Tree individual', linewidth=2)
plt.xlabel('Años de Experiencia')
plt.ylabel('Salario')
plt.title('Decision Tree individual (Overfitting)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(df['YearsExperience'], df['Salary'], label='Datos reales', alpha=0.7)
plt.plot(x_plot, rf_predictions, color='green', label='Random Forest', linewidth=2)
plt.xlabel('Años de Experiencia')
plt.ylabel('Salario')
plt.title('Random Forest (Predicción suavizada)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Comparar métricas
train_pred_tree = tree_reg_full.predict(df[["YearsExperience"]])
train_pred_rf = rf_regressor.predict(df[["YearsExperience"]])

print(f"Decision Tree individual - MSE: {mean_squared_error(df['Salary'], train_pred_tree):.2f}, R²: {r2_score(df['Salary'], train_pred_tree):.4f}")
print(f"Random Forest - MSE: {mean_squared_error(df['Salary'], train_pred_rf):.2f}, R²: {r2_score(df['Salary'], train_pred_rf):.4f}")


¡Observa cómo el Random Forest produce una curva de predicción mucho más suavizada en comparación con la curva irregular con overfitting de un único decision tree!


### Importancia de features

Una de las grandes ventajas de los Random Forests es que pueden indicarnos qué features son más importantes para hacer predicciones. Esto se calcula midiendo cuánto reduce cada feature la impureza (Gini o entropía) en todos los árboles.


In [ ]:
# Importancia de features para la clasificación iris
import pandas as pd
import matplotlib.pyplot as plt

importances = rf_classifier.feature_importances_
feature_names = iris.feature_names

# Crear un dataframe para mejor visualización
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Importancia de features:")
print(feature_importance_df)

# Visualizar
plt.figure(figsize=(8, 5))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importancia')
plt.title('Importancia de Features en Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Hiperparámetros importantes

Los Random Forests tienen varios hiperparámetros que se pueden ajustar:

| Parámetro | Descripción | Valor por defecto | Impacto |
|-----------|-------------|---------|--------|
| `n_estimators` | Número de árboles en el bosque | 100 | Más árboles = mejor rendimiento pero más lento. Normalmente 100-500 es suficiente |
| `max_depth` | Profundidad máxima de cada árbol | None (ilimitado) | Controla el overfitting. Valores más bajos previenen el overfitting |
| `min_samples_split` | Mín. de muestras para dividir un nodo | 2 | Valores más altos previenen el overfitting |
| `min_samples_leaf` | Mín. de muestras requeridas en un nodo hoja | 1 | Valores más altos crean modelos más suavizados |
| `max_features` | Máx. de features a considerar por división | 'sqrt' (clasificación)<br>'1.0' (regresión) | Valores más bajos aumentan la diversidad entre árboles |
| `random_state` | Semilla para reproducibilidad | None | Establecer a un número para resultados reproducibles |
| `n_jobs` | Número de núcleos de CPU a usar | None (1 núcleo) | Establecer a -1 para usar todos los núcleos disponibles |


### Ventajas y desventajas de los Random Forests

**Ventajas:**
- **Reduce el overfitting:** Al promediar múltiples árboles, la varianza se reduce significativamente
- **Alta precisión:** A menudo logra un excelente rendimiento sin configuración adicional
- **Maneja datos faltantes:** Puede mantener la precisión incluso con valores faltantes
- **Funciona con features categóricas y numéricas:** Requiere preprocesamiento mínimo
- **Importancia de features:** Proporciona información sobre qué features importan más
- **Robusto frente a outliers:** Los árboles individuales pueden hacer overfitting de outliers, pero el ensemble los promedia
- **Paralelizable:** Los árboles se pueden entrenar de forma independiente en múltiples núcleos de CPU

**Desventajas:**
- **Menos interpretable:** No se pueden visualizar cientos de árboles como uno solo
- **Predicción más lenta:** Debe agregar predicciones de muchos árboles
- **Mayor tamaño del modelo:** Requiere almacenar múltiples árboles en memoria
- **No ideal para extrapolación:** No puede predecir más allá del rango de los datos de training
- **Puede hacer overfitting con datos ruidosos:** Si los árboles son demasiado profundos y el dataset es muy ruidoso


### Resumen: Decision Trees vs. Random Forests

| Aspecto | Decision Tree individual | Random Forest |
|--------|---------------------|---------------|
| **Tipo de modelo** | Modelo único | Ensemble de modelos |
| **Overfitting** | Alta tendencia al overfitting | Overfitting reducido |
| **Varianza** | Alta varianza | Baja varianza |
| **Bias** | Bajo bias (si es profundo) | Bias ligeramente mayor |
| **Interpretabilidad** | Altamente interpretable | Menos interpretable |
| **Tiempo de training** | Rápido | Más lento (entrena múltiples árboles) |
| **Tiempo de predicción** | Muy rápido | Más lento (agrega predicciones) |
| **Accuracy** | Menor (especialmente en datos de test) | Mayor (más robusto) |
| **Cuándo usarlo** | Se necesita interpretabilidad, dataset pequeño | Se necesita precisión, se puede asumir el coste computacional |

**Conclusión clave:** Los Random Forests intercambian algo de interpretabilidad y velocidad por una generalización y precisión significativamente mejores. Son uno de los algoritmos de machine learning más potentes y ampliamente usados en la práctica.


### Recursos adicionales sobre Random Forests

- [Random Forests - Documentación de Scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [StatQuest: Random Forests Part 1 - Building, Using and Evaluating](https://www.youtube.com/watch?v=J4Wdy0Wc_xQ)
- [StatQuest: Random Forests Part 2 - Missing Data and Clustering](https://www.youtube.com/watch?v=sQ870aTKqiM)
- Artículo original: Breiman, L. (2001). "Random Forests". Machine Learning. 45 (1): 5–32.
